<a href="https://colab.research.google.com/github/harsh3-web/Ecommerce-Recommender/blob/main/notebooks/01_data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Data generation

Builds the dataset: **200 real Myntra products** (25 per category, with photos) from the *Fashion Product Images (Small)* dataset, plus **500 simulated users** who each secretly like 2 categories. Simulates browsing (80% in liked categories) and purchases (35% vs 10% buy probability), with long-tail product popularity. Saves 6 CSVs and 200 images to Google Drive. Seed 42 → fully reproducible.

In [ ]:
import os
if not os.path.exists("/content/Ecommerce-Recommender"):          # clone only once
    !git clone -q https://github.com/harsh3-web/Ecommerce-Recommender.git /content/Ecommerce-Recommender
%cd /content/Ecommerce-Recommender
!pip install -q datasets

In [ ]:
import numpy as np
import pandas as pd

SEED = 42  # to fix the starting point of the random number generator
rng = np.random.default_rng(SEED)
N_USERS = 500  #customers our fake shop has
N_PRODUCT = 200 #how many products are in the catalogue
START_DATE = pd.Timestamp("2026-03-01")
N_DAYS = 180


In [ ]:
# Price range (in rupees) for each of our 8 categories.
# The dataset has no prices, so price and rating are simulated from these ranges.
CATEGORIES = {
    "T-Shirts": {"price": (299, 1299)},
    "Shirts":   {"price": (599, 2499)},
    "Jeans":    {"price": (799, 3499)},
    "Dresses":  {"price": (699, 3999)},
    "Jackets":  {"price": (1299, 5999)},
    "Shoes":    {"price": (999, 6999)},
    "Watches":  {"price": (999, 9999)},
    "Bags":     {"price": (499, 3999)},
}

In [ ]:
print(len(CATEGORIES), "categories:", list(CATEGORIES))

8 categories: ['T-Shirts', 'Shirts', 'Jeans', 'Dresses', 'Jackets', 'Shoes', 'Watches', 'Bags']


In [ ]:
# Cities for the simulated users
CITIES = ["Delhi", "Mumbai", "Bengaluru", "Kolkata", "Chennai", "Hyderabad", "Pune", "Jaipur"]

cat_names = list(CATEGORIES)     # the 8 category names, used for user tastes

In [ ]:
## Helper function: simulated price and rating

def price_and_rating(info):
  low, high = info["price"]

  price = int(rng.integers(low,high)// 100*100+99)   ## shop-style prices like 499, 1499, 2399
  rating = np.clip(rng.normal(4.0,0.5),1.0,5.0)    ## normal distribution around 4.0 (std 0.5), clipped to 1-5

  return price, round(float(rating),1)

In [ ]:
# 'datasets' downloads ready-made datasets from Hugging Face (no Kaggle API key needed)
!pip install -q datasets

In [ ]:
from datasets import load_dataset

# ~270 MB download: 44k Myntra products, each with metadata + an 80x60 product photo
ds = load_dataset("ashraq/fashion-product-images-small", split="train")
print(ds)          # shows the column names and the number of rows

Dataset({
    features: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image'],
    num_rows: 44072
})


In [ ]:
# remove_columns("image") -> keep only the text columns, so the pandas table stays small and fast
meta = ds.remove_columns("image").to_pandas()
meta["row"] = np.arange(len(meta))     # remember each product's position in ds, to fetch its image later
print(meta.shape)
print(meta["articleType"].value_counts().head(25))   # the 25 most common product types

(44072, 11)
articleType
Tshirts                  7065
Shirts                   3212
Casual Shoes             2845
Watches                  2542
Sports Shoes             2016
Kurtas                   1844
Tops                     1762
Handbags                 1759
Heels                    1323
Sunglasses               1073
Wallets                   928
Flip Flops                914
Sandals                   895
Briefs                    847
Belts                     813
Backpacks                 724
Socks                     686
Formal Shoes              637
Perfume and Body Mist     603
Jeans                     602
Shorts                    547
Trousers                  530
Flats                     500
Bra                       477
Dresses                   464
Name: count, dtype: int64


In [ ]:
# Myntra's 'articleType' -> our category name (some of our categories combine 2 types)
TYPE_TO_CAT = {
    "Tshirts": "T-Shirts", "Shirts": "Shirts", "Jeans": "Jeans", "Dresses": "Dresses",
    "Jackets": "Jackets", "Casual Shoes": "Shoes", "Sports Shoes": "Shoes",
    "Watches": "Watches", "Backpacks": "Bags", "Handbags": "Bags",
}
meta = meta[meta["articleType"].isin(TYPE_TO_CAT)].copy()      # keep only these types
meta["category"] = meta["articleType"].map(TYPE_TO_CAT)         # add our category name
meta = meta.dropna(subset=["productDisplayName", "baseColour", "usage", "gender", "season"])  # drop incomplete rows
print(meta["category"].value_counts())                          # each category needs at least 25

category
T-Shirts    7065
Shoes       4861
Shirts      3212
Watches     2542
Bags        2483
Jeans        602
Dresses      464
Jackets      258
Name: count, dtype: int64


In [ ]:
# groupby("category").sample(n=25) -> 25 random rows from EACH category
# random_state=SEED -> the same 25 products every time you run it
picked = meta.groupby("category").sample(n=25, random_state=SEED)
picked = picked.sort_values("category").reset_index(drop=True)    # tidy order, rows 0..199
print(picked.shape)                                                 # (200, ...)
picked[["productDisplayName", "category", "baseColour", "usage"]].head()

(200, 12)


,productDisplayName,category,baseColour,usage
0,Spice Art Women Casual Mouse Brown HandBag,Bags,Brown,Casual
1,Peperone Women Beige Handbag,Bags,Beige,Casual
2,Wrangler Unisex Canvas Khaki Sling Bag,Bags,Khaki,Casual
3,Murcia Women Casual Brown Handbag,Bags,Brown,Casual
4,Kiara Women Rose Pink Handbag,Bags,Pink,Casual


In [ ]:
def make_real_product(pid, row):
    """One row of our products table, built from a real Myntra product."""
    price, rating = price_and_rating(CATEGORIES[row.category])   # dataset has no price/rating -> simulate
    desc = f"{row.baseColour} {row.articleType} for {row.gender}, {row.usage.lower()} wear for {row.season}."
    return {"product_id": pid, "product_name": row.productDisplayName, "category": row.category,
            "description": desc, "price": price, "rating": rating,
            "source_id": int(row.id), "hf_row": int(row.row)}   # keep links back to the original data

In [ ]:
# itertuples() goes through rows; each row's columns are accessed like row.category
# enumerate gives a counter i = 0, 1, 2... -> product ids 1..200
products = pd.DataFrame([make_real_product(i + 1, r) for i, r in enumerate(picked.itertuples())])
print(products.shape)
print(products["category"].value_counts())        # 25 each
products[["product_id", "product_name", "category", "description"]].head()

(200, 8)
category
Bags        25
Dresses     25
Jackets     25
Jeans       25
Shirts      25
Shoes       25
T-Shirts    25
Watches     25
Name: count, dtype: int64


,product_id,product_name,category,description
0,1,Spice Art Women Casual Mouse Brown HandBag,Bags,"Brown Handbags for Women, casual wear for Winter."
1,2,Peperone Women Beige Handbag,Bags,"Beige Handbags for Women, casual wear for Summer."
2,3,Wrangler Unisex Canvas Khaki Sling Bag,Bags,"Khaki Handbags for Unisex, casual wear for Sum..."
3,4,Murcia Women Casual Brown Handbag,Bags,"Brown Handbags for Women, casual wear for Winter."
4,5,Kiara Women Rose Pink Handbag,Bags,"Pink Handbags for Women, casual wear for Winter."


In [ ]:
import os
os.makedirs("datasets/images", exist_ok=True)     # create the folder (no error if it already exists)

for pid, hf_row in zip(products["product_id"], products["hf_row"]):
    img = ds[int(hf_row)]["image"].convert("RGB")             # fetch the photo; RGB = 3 colour channels
    img.save(f"datasets/images/{pid}_1_front.jpg")            # filename format: <product_id>_1_front.jpg
print(len(os.listdir("datasets/images")), "images saved")     # should print 200

200 images saved


In [ ]:
## build the users table

users = pd.DataFrame({
    "user_id": np.arange(1, N_USERS + 1),
    "age": rng.integers(18,56, N_USERS),
    "gender": rng.choice(["M","F"], N_USERS),
    "city": rng.choice(CITIES, N_USERS),
})

users.head()

,user_id,age,gender,city
0,1,27,F,Pune
1,2,28,F,Chennai
2,3,20,F,Delhi
3,4,23,M,Jaipur
4,5,34,M,Bengaluru


In [ ]:
## Lookup tables and popularity
pid_to_cat = dict(zip(products["product_id"], products["category"]))  ## e.g. is product 37 a Shoe?

# Pareto distribution = long tail: most products get small values, a few get very large ones
popularity = rng.pareto(1.5, N_PRODUCT) + 1

pop = dict(zip(products["product_id"], popularity)) # product_id -> popularity weight

# category -> array of its product ids; and an array of ALL product ids
ids_by_cat = {c: g["product_id"].to_numpy() for c, g in products.groupby("category")}
all_ids = products["product_id"].to_numpy()


In [ ]:
#Pick a product and a random time
def sample_product(pool):
    """Pick ONE product from 'pool', where popular products are more likely to be picked."""
    weights = np.array([pop[p] for p in pool])    # popularity of each product in the pool
    return int(rng.choice(pool, p=weights / weights.sum()))    # p = probabilities, must sum to 1

def random_time():
    minutes = int(rng.integers(0, N_DAYS * 24 * 60))     # 180 days converted to minutes
    return START_DATE + pd.Timedelta(minutes=minutes)    # Timedelta = a duration you can add to a date

In [ ]:
##Decide whether the user buys

def maybe_buy(uid, pid, t, liked):
    """After browsing product 'pid' at time 't', maybe the user buys it."""
    buy_prob = 0.35 if pid_to_cat[pid] in liked else 0.10   # liked category -> 35% chance, else 10%
    if rng.random() >= buy_prob:                             # rng.random() = random number between 0 and 1
        return None                                          # no purchase
    buy_time = t + pd.Timedelta(hours=int(rng.integers(1, 72)))   # buy 1 to 72 hours after browsing
    return {"user_id": uid, "product_id": pid, "timestamp": buy_time,
            "quantity": int(rng.integers(1, 3))}             # quantity 1 or 2 (upper limit is excluded)

In [ ]:
def one_browse(uid, liked, seen):
    # Most of the time, pick a product from a category the user likes
    if rng.random() < 0.8:
        category = rng.choice(liked)
        pool = ids_by_cat[str(category)]
    else:
        # Sometimes let the user explore other categories
        pool = all_ids

    # Pick a random product
    pid = sample_product(pool)

    # Skip the product if the user has already seen it
    if pid in seen:
        return None, None

    seen.add(pid)

    # Generate a random time for the browsing event
    t = random_time()

    browse = {
        "user_id": uid,
        "product_id": pid,
        "timestamp": t
    }

    # Check if the user ends up buying the product
    purchase = maybe_buy(uid, pid, t, liked)

    return browse, purchase

In [ ]:
def simulate_user(uid):
    # Pick 2 different categories that the user likes
    liked = [str(c) for c in rng.choice(cat_names, size=2, replace=False)]

    browses = []
    buys = []
    seen = set()

    # Simulate the user's browsing activity
    num_browses = int(rng.poisson(12)) + 4

    for _ in range(num_browses):
        browse, purchase = one_browse(uid, liked, seen)

        if browse:
            browses.append(browse)

        if purchase:
            buys.append(purchase)

    return liked, browses, buys

In [ ]:
def ensure_one_purchase(uid, browses, buys):
    # If the user didn't buy anything, use their first viewed product
    if len(buys) == 0:
        first = browses[0]

        purchase = {
            "user_id": uid,
            "product_id": first["product_id"],
            "timestamp": first["timestamp"] + pd.Timedelta(hours=5),
            "quantity": 1
        }

        buys.append(purchase)

    return buys

In [ ]:
tastes = []
all_browses = []
all_buys = []

for uid in range(1, N_USERS + 1):
    liked, browses, buys = simulate_user(uid)

    # Save the user's preferred categories
    tastes.append({
        "user_id": uid,
        "liked_category_1": liked[0],
        "liked_category_2": liked[1]
    })

    # Add this user's browsing and purchase data
    all_browses.extend(browses)
    all_buys.extend(ensure_one_purchase(uid, browses, buys))

print("Done!")
print("Total browses:", len(all_browses))
print("Total purchases:", len(all_buys))

Done!
Total browses: 6269
Total purchases: 1901


In [ ]:
# Convert the browsing and purchase lists into DataFrames
browsing = pd.DataFrame(all_browses)
purchases = pd.DataFrame(all_buys)

# Put the events in chronological order
browsing = browsing.sort_values("timestamp").reset_index(drop=True)
purchases = purchases.sort_values("timestamp").reset_index(drop=True)

# Store each user's two preferred categories
tastes = pd.DataFrame(tastes)

# Check the first few purchase records
purchases.head()

,user_id,product_id,timestamp,quantity
0,217,186,2026-03-02 12:59:00,2
1,101,114,2026-03-02 13:49:00,1
2,179,34,2026-03-02 18:39:00,1
3,354,162,2026-03-03 00:19:00,1
4,43,34,2026-03-03 01:02:00,2


In [ ]:
# Create image paths for each product
product_images = pd.DataFrame({
    "product_id": products["product_id"],
    "image_path": [
        f"datasets/images/{pid}_1_front.jpg"
        for pid in products["product_id"]
    ]
})

product_images.head()

,product_id,image_path
0,1,datasets/images/1_1_front.jpg
1,2,datasets/images/2_1_front.jpg
2,3,datasets/images/3_1_front.jpg
3,4,datasets/images/4_1_front.jpg
4,5,datasets/images/5_1_front.jpg


In [ ]:
# index=False -> don't save pandas' row numbers as an extra column
# File names must match what the other notebooks and the app read
users.to_csv("users_expanded.csv", index=False)
products.to_csv("products_expanded.csv", index=False)
product_images.to_csv("product_images_expanded.csv", index=False)
purchases.to_csv("purchases_expanded.csv", index=False)
browsing.to_csv("browsing_history_expanded.csv", index=False)
tastes.to_csv("ground_truth_tastes.csv", index=False)   # ONLY for checking results, never given to a model

In [ ]:
!pwd                 # pwd = "print working directory" -> shows the current folder
!ls -lh *.csv        # lists the 6 CSV files saved there, with their sizes

/content/Ecommerce-Recommender/Ecommerce-Recommender/Ecommerce-Recommender/Ecommerce-Recommender
-rw-r--r-- 1 root root 167K Oct  2 09:07 browsing_history_expanded.csv
-rw-r--r-- 1 root root 8.9K Oct  2 09:07 ground_truth_tastes.csv
-rw-r--r-- 1 root root 6.9K Oct  2 09:07 product_images_expanded.csv
-rw-r--r-- 1 root root  23K Oct  2 09:07 products_expanded.csv
-rw-r--r-- 1 root root  55K Oct  2 09:07 purchases_expanded.csv
-rw-r--r-- 1 root root 8.1K Oct  2 09:07 users_expanded.csv


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/Ecommerce-Recommender-data"

!mkdir -p "{DATA}/images"
!cp *.csv "{DATA}/"                         # overwrite the old CSVs with the new real-product data
!cp datasets/images/*.jpg "{DATA}/images/"  # save the 200 images too
!ls "{DATA}/images" | wc -l                 # wc -l counts lines -> should print 200

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
200
